# Gold-standard (Cell Ranger) — expression consistency after mapping (marketing appendix)

This notebook is an *optional* deep dive that turns the gold-standard pipeline into a stronger marketing artifact.

## Rationale

The Fig 4c notebook focuses on **feature-set consistency** (gene identity) across Ensembl releases under a fixed snapshot boundary.
This companion notebook asks a complementary question:

> After mapping each release-specific feature space into a single target release, do the **pseudo-bulk expression profiles** become consistent across releases?

This is not an accuracy benchmark; it is a sanity/marketing visualization that the mapped feature space behaves coherently.

## Inputs and prerequisites

- `.h5ad` files produced by `experiment_cellranger_idtrack/create_data.ipynb`
- Conversion caches produced by `experiment_cellranger_idtrack/analysis_gold_standard_fig4c.ipynb`

## Outputs

- `idtrack-manuscript/figures/fig_gold_standard_pseudobulk_correlation.pdf`
- Cached per-release pseudo-bulk vectors under `idtrack/docs/_notebooks/idtrack_cache/experiments/gold_standard_cellranger/`


In [ ]:
from __future__ import annotations

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import sys

# Add experiments/src to sys.path
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    experiments_cache_dir,
    idtrack_cache_dir,
    manuscript_figures_dir,
    read_pickle,
    write_pickle,
)

try:
    import anndata as ad
except Exception as e:  # noqa: S110
    raise ImportError('This notebook requires anndata/scanpy environment.') from e

if sns is not None:
    sns.set_theme(style='whitegrid', context='paper')

plt.rcParams.update({'savefig.dpi': 300, 'figure.dpi': 140})

ANNDATA_DIR_RAW = os.environ.get('GOLD_STANDARD_ANNDATA_DIR', '').strip()
ANNDATA_DIR = Path(ANNDATA_DIR_RAW).expanduser().resolve() if ANNDATA_DIR_RAW else None

IDTRACK_LOCAL_REPO = idtrack_cache_dir(REPO_ROOT)
CACHE_DIR = experiments_cache_dir(REPO_ROOT, experiment='gold_standard_cellranger')
MANUSCRIPT_FIGURES = manuscript_figures_dir(REPO_ROOT)

print('ANNDATA_DIR:', ANNDATA_DIR)
print('CACHE_DIR:', CACHE_DIR)
print('MANUSCRIPT_FIGURES:', MANUSCRIPT_FIGURES)


In [ ]:
# -------------------- Configuration --------------------

TARGET_RELEASE = 114
FINAL_DATABASE = None  # keep Ensembl backbone
STRATEGY = 'best'      # use 1→1 for stable pseudo-bulk alignment

# How to reduce each AnnData into a vector for correlation.
# - 'sum' is pseudo-bulk (recommended)
PSEUDOBULK_MODE = 'sum'

print('TARGET_RELEASE:', TARGET_RELEASE)
print('PSEUDOBULK_MODE:', PSEUDOBULK_MODE)


In [ ]:
# -------------------- Discover .h5ad files and pick dataset/assembly --------------------

if not ANNDATA_DIR or not ANNDATA_DIR.exists():
    raise FileNotFoundError('Set GOLD_STANDARD_ANNDATA_DIR to the folder containing generated .h5ad files.')

h5ads = sorted(ANNDATA_DIR.glob('*.h5ad'))
pat = re.compile(r'^(?P<dataset>.+?)_(?P<assembly>[^_]+)_(?P<release>\d+)\.h5ad$')

rows = []
for p in h5ads:
    m = pat.match(p.name)
    if not m:
        continue
    rows.append({'dataset': m.group('dataset'), 'assembly': m.group('assembly'), 'release': int(m.group('release')), 'path': str(p)})

files = pd.DataFrame(rows).sort_values(['dataset', 'assembly', 'release']).reset_index(drop=True)
if files.empty:
    raise RuntimeError('No .h5ad files matched the expected pattern: <dataset>_<assembly>_<release>.h5ad')

grp = files.groupby(['dataset', 'assembly']).size().reset_index(name='n_files')
best = grp.sort_values('n_files', ascending=False).iloc[0]
dataset = str(best['dataset'])
assembly = str(best['assembly'])

subset = files[(files['dataset'] == dataset) & (files['assembly'] == assembly)].sort_values('release').reset_index(drop=True)
print('Selected dataset:', dataset)
print('Selected assembly:', assembly)
print('Releases:', subset['release'].tolist())

subset.head()


In [ ]:
# -------------------- Load conversion caches and compute pseudo-bulk vectors --------------------

def _safe_stem(s: str) -> str:
    return ''.join(c if c.isalnum() or c in {'-', '_'} else '_' for c in str(s))


def _conv_pickle_path(dataset: str, assembly: str, release: int) -> Path:
    return CACHE_DIR / (
        f"idtrack_matchings_{_safe_stem(dataset)}_{_safe_stem(assembly)}_from{release}_to{TARGET_RELEASE}"
        f"_final{FINAL_DATABASE or 'ensembl'}_strategy{STRATEGY}.pickle"
    )


def load_matchings(dataset: str, assembly: str, release: int) -> list[dict]:
    p = _conv_pickle_path(dataset, assembly, release)
    if not p.exists():
        raise FileNotFoundError(
            f"Missing conversion cache for release {release}: {p}. "
            "Run `analysis_gold_standard_fig4c.ipynb` to generate caches."
        )
    obj = read_pickle(p)
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict) and 'matchings' in obj:
        return obj['matchings']
    raise TypeError(f'Unexpected conversion cache format in {p}: {type(obj)}')


def mapping_1_to_1(matchings: list[dict]) -> dict[str, str]:
    out: dict[str, str] = {}
    for m in matchings:
        if m.get('no_corresponding') or m.get('no_conversion') or m.get('no_target'):
            continue
        t = m.get('target_id') or []
        if len(t) != 1:
            continue
        q = str(m.get('query_id'))
        out[q] = str(t[0])
    return out


def pseudobulk_target_vector(adata, q_to_t: dict[str, str]) -> pd.Series:
    # Build per-gene sums in the source space
    if PSEUDOBULK_MODE != 'sum':
        raise ValueError(f'Unsupported PSEUDOBULK_MODE: {PSEUDOBULK_MODE}')

    X = adata.X
    sums = X.sum(axis=0)
    if hasattr(sums, 'A1'):
        sums = sums.A1
    sums = np.asarray(sums).ravel()

    src_ids = [str(x) for x in adata.var_names]
    df = pd.DataFrame({'query_id': src_ids, 'sum': sums})
    df['target_id'] = df['query_id'].map(q_to_t)
    df = df.dropna(subset=['target_id'])

    # Collapse potential n→1 collisions by summing into target space
    vec = df.groupby('target_id')['sum'].sum()
    vec.name = 'sum'
    return vec


vectors: dict[int, pd.Series] = {}
for r in subset.itertuples(index=False):
    rel = int(r.release)
    out_pkl = CACHE_DIR / f"pseudobulk_{_safe_stem(dataset)}_{_safe_stem(assembly)}_{rel}_to{TARGET_RELEASE}.pickle"
    if out_pkl.exists():
        vectors[rel] = read_pickle(out_pkl)
        continue

    adata = ad.read_h5ad(str(r.path))
    matchings = load_matchings(dataset, assembly, rel)
    q_to_t = mapping_1_to_1(matchings)
    vec = pseudobulk_target_vector(adata, q_to_t)
    write_pickle(vec, out_pkl)
    vectors[rel] = vec

print('Pseudo-bulk vectors:', len(vectors))
list(vectors)[:10]


In [ ]:
# -------------------- Build correlation matrix and plot --------------------

releases = sorted(vectors)
if len(releases) < 2:
    raise RuntimeError('Need at least 2 releases to compute a correlation matrix.')

# Align on intersection of target genes across releases for a clean correlation.
common = set(vectors[releases[0]].index)
for r in releases[1:]:
    common &= set(vectors[r].index)

common = sorted(common)
print('Common target genes:', len(common))

mat = []
for r in releases:
    v = vectors[r].reindex(common).fillna(0.0).astype(float)
    mat.append(np.log1p(v.values))

X = np.vstack(mat)
corr = np.corrcoef(X)
corr_df = pd.DataFrame(corr, index=releases, columns=releases)

fig, ax = plt.subplots(1, 1, figsize=(6.2, 5.2))
if sns is not None:
    sns.heatmap(corr_df, ax=ax, cmap='Blues', vmin=0, vmax=1, square=True, cbar_kws={'label': 'Pearson r'})
else:
    im = ax.imshow(corr_df.values, cmap='Blues', vmin=0, vmax=1)
    fig.colorbar(im, ax=ax, label='Pearson r')
    ax.set_xticks(range(len(releases)))
    ax.set_yticks(range(len(releases)))
    ax.set_xticklabels(releases, rotation=30, ha='right')
    ax.set_yticklabels(releases)

ax.set_title('Gold standard: pseudo-bulk consistency after 1→1 mapping')
ax.set_xlabel('Starting release')
ax.set_ylabel('Starting release')
fig.tight_layout()

out_fig = MANUSCRIPT_FIGURES / 'fig_gold_standard_pseudobulk_correlation.pdf'
fig.savefig(out_fig, bbox_inches='tight')
print('Saved:', out_fig)

corr_df
